In [ ]:
# =====================================================================================
# セル1: ライブラリの準備とデータ読み込み (CH03・CH04対応・最前面ダイアログ版)
# =====================================================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
import io
import re
import tkinter as tk
from tkinter import filedialog
import numpy as np

# --- ファイルを読み込んでカラム名を変更するための関数 ---
def load_and_rename_columns(file_path: str) -> pd.DataFrame:
    NEW_COLUMN_NAMES = [
        '元ファイル', 'キャビ厚', '射出速度[mm/s]',
        'CH03_立ち上がり時間(s)', 'CH03_立ち上がり圧力(MPa)',
        'CH04_立ち上がり時間(s)', 'CH04_立ち上がり圧力(MPa)',
        'CH03圧力@CH04立ち上がり時(MPa)'
    ]
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='shift_jis')

    if len(df.columns) == len(NEW_COLUMN_NAMES):
        df.columns = NEW_COLUMN_NAMES
    else:
        print(f"警告: {os.path.basename(file_path)} のカラム数が期待値と異なります。カラム名の変更をスキップします。")
    return df

# --- ステップ1: ファイルの選択と読み込み ---
print("--- ステップ1: ファイル選択と読み込み ---")

# tkinterの設定
root = tk.Tk()
root.withdraw() # メインウィンドウを非表示にする

# ★追加箇所: ダイアログを強制的に最前面（常に手前）に表示させる設定
root.call('wm', 'attributes', '.', '-topmost', True)

file_paths = filedialog.askopenfilenames(
    title="解析したいCSVファイルを選択してください",
    filetypes=[("CSVファイル", "*.csv"), ("すべてのファイル", "*.*")]
)

if not file_paths:
    print("ファイルが選択されませんでした。処理を中断します。")
else:
    print(f"{len(file_paths)}個のファイルが選択されました。")
    dataframes = {}
    file_path_map = {}
    
    for path in file_paths:
        file_name = os.path.basename(path)
        dataframes[file_name] = load_and_rename_columns(path)
        file_path_map[file_name] = os.path.dirname(path) 
        
    print("すべてのファイルの読み込みと前処理が完了しました。\n")
    
    # ★追加箇所: 読み込んだファイルの名前とパスをプリントする
    print("--- 読み込んだファイル一覧 ---")
    for path in file_paths:
        print(f"ファイル名: {os.path.basename(path)}")
        print(f"ファイルパス: {path}")
        print("-" * 50)

In [2]:
# =====================================================================================
# セル2: 物理量および統計量(線形/対数)の計算
# =====================================================================================

if 'dataframes' in locals() and dataframes:
    print("\n--- ステップ2: データ処理（全計算）---")

    for key, df in dataframes.items():
        print(f"  計算中: {key}")

        # --- 10ショット目の存在チェックとデータ抽出・複製 ---
        error_found = False
        new_rows = []
        for i in range(0, len(df), 10):
            chunk = df.iloc[i:i+10]
            
            if len(chunk) < 10:
                print(f"【致命的なエラー】{i+1}行目からの条件ブロックが10ショットに満たない（データ不足）です。")
                error_found = True
                break
                
            if chunk['射出速度[mm/s]'].nunique() > 1:
                print(f"【致命的なエラー】{i+1}行目付近の10個のデータ内で射出速度の条件が一致していません。")
                error_found = True
                break
                
            shot_10th = chunk.iloc[9]
            for _ in range(10):
                new_rows.append(shot_10th.copy())
                
        if error_found:
            raise ValueError(f"ファイル '{key}' 内に10ショット目が存在しない、または形式が不正な条件が含まれています。処理を中断します。")
            
        print("  ✓ 全条件で10ショット目の存在を確認しました。10ショット目のデータのみを使用して解析を開始します。")
        df = pd.DataFrame(new_rows).reset_index(drop=True)
        # ---------------------------------------------------

        df['キャビ厚'] = pd.to_numeric(df['キャビ厚'].astype(str).str.replace('mm', ''), errors='coerce')
        cavity_thickness_numeric = df['キャビ厚']
        distance_L34 = 17.5   

        df['時間差ΔT34(s)'] = df['CH04_立ち上がり時間(s)'] - df['CH03_立ち上がり時間(s)']
        df['圧力差ΔP34(Pa)'] = (df['CH03圧力@CH04立ち上がり時(MPa)'] - df['CH04_立ち上がり圧力(MPa)']) * 1000000
        df['流速V34(mm/s)'] = distance_L34 / df['時間差ΔT34(s)']
        df['流路断面積(mm²)'] = cavity_thickness_numeric * 40
        df['流量Q34(mm³/s)'] = df['流速V34(mm/s)'] * df['流路断面積(mm²)']
        df['樹脂粘度η34(Pa・s)'] = (df['圧力差ΔP34(Pa)'] * df['流路断面積(mm²)'] * cavity_thickness_numeric**2) / (12 * distance_L34 * df['流量Q34(mm³/s)'])
        df['せん断速度γ34(1/s)'] = (6 * df['流量Q34(mm³/s)']) / (df['流路断面積(mm²)'] * cavity_thickness_numeric)
        df['温度'] = pd.to_numeric(df['元ファイル'].str.extract(r'(\d+)℃')[0], errors='coerce')

        # --- 各列の平均・標準偏差の計算 (10行ごと) ---
        group_id_by_position = df.index // 10
        linear_cols = ['時間差ΔT34(s)', '流速V34(mm/s)', '流量Q34(mm³/s)', '圧力差ΔP34(Pa)']
        log_cols = ['せん断速度γ34(1/s)', '樹脂粘度η34(Pa・s)']

        for col in linear_cols:
            df[f'{col}_平均'] = df.groupby(group_id_by_position)[col].transform('mean')
            # 過去の指示に基づき、平均基準ではなく絶対値シフトとして計算
            df[f'{col}_標準偏差上'] = df.groupby(group_id_by_position)[col].transform('std')
            df[f'{col}_標準偏差下'] = -df.groupby(group_id_by_position)[col].transform('std')

        for col in log_cols:
            # 幾何平均・幾何標準偏差 (対数空間で計算して戻す)
            log_val = np.log10(df[col].replace(0, np.nan))
            log_mean = log_val.groupby(group_id_by_position).transform('mean')
            log_std = log_val.groupby(group_id_by_position).transform('std')
            df[f'{col}_幾何平均'] = 10 ** log_mean
            df[f'{col}_幾何標準偏差上'] = 10 ** (log_mean + log_std)
            df[f'{col}_幾何標準偏差下'] = 10 ** (log_mean - log_std)

        dataframes[key] = df
    print("すべての計算が完了しました。")
else:
    print("エラー: ステップ1が実行されていません。")


--- ステップ2: データ処理（全計算）---
  計算中: NR_summary_rise_start_results.csv
  ✓ 全条件で10ショット目の存在を確認しました。10ショット目のデータのみを使用して解析を開始します。
すべての計算が完了しました。


In [3]:
# =====================================================================================
# セル3-1: 集約データのCSV書き出し
# =====================================================================================

print("\n--- ステップ3: 集約データのCSV書き出し ---")

if 'dataframes' in locals() and 'file_path_map' in locals():
    for key, df in dataframes.items():
        output_folder = file_path_map[key] 
        group_id_by_position = df.index // 10
        
        columns_to_aggregate = ['キャビ厚', '温度', '射出速度[mm/s]']
        for col in ['時間差ΔT34(s)', '流速V34(mm/s)', '流量Q34(mm³/s)', '圧力差ΔP34(Pa)']:
            columns_to_aggregate.extend([f'{col}_平均', f'{col}_標準偏差上', f'{col}_標準偏差下'])
        for col in ['せん断速度γ34(1/s)', '樹脂粘度η34(Pa・s)']:
            columns_to_aggregate.extend([f'{col}_幾何平均', f'{col}_幾何標準偏差上', f'{col}_幾何標準偏差下'])
        
        aggregated_df = df.groupby(group_id_by_position)[columns_to_aggregate].first().reset_index(drop=True)
        
        base_name = os.path.splitext(key)[0]
        output_filename_leaf = f"{base_name}_aggregated_summary.csv"
        output_full_path = os.path.join(output_folder, output_filename_leaf)
        
        try:
            aggregated_df.to_csv(output_full_path, index=False, encoding='utf-8-sig')
            print(f"  完了: {output_full_path} に保存しました。") 
        except Exception as e:
            print(f"  エラー: {key} のCSV保存に失敗しました。 {e}")


--- ステップ3: 集約データのCSV書き出し ---
  完了: C:/Users/0uh2j/OneDrive/Desktop/実験データ2026/本実験データ/算出グラフ_ローパス条件比較/カットオフ周波数80閾値0.4\NR_summary_rise_start_results_aggregated_summary.csv に保存しました。


In [4]:
# =====================================================================================
# セル3-2: 計算結果の途中確認
# =====================================================================================

if 'dataframes' in locals() and dataframes:
    print("\n--- ステップ3: 最終データの確認 ---")
    first_df_key = list(dataframes.keys())[0]
    final_df = dataframes[first_df_key]

    if len(final_df.columns) > 10:
        display_df = pd.concat([final_df.iloc[:, :4], final_df.iloc[:, -4:]], axis=1)
        print("\n【データの最初の4列と最後の4列】")
        print(display_df.head())
    else:
        print("エラー：計算が正常に完了したか確認してください。")


--- ステップ3: 最終データの確認 ---

【データの最初の4列と最後の4列】
                    元ファイル  キャビ厚  射出速度[mm/s]  CH03_立ち上がり時間(s)  \
0  010_190℃_020_0.5mm.csv   0.5          20         0.779205   
1  010_190℃_020_0.5mm.csv   0.5          20         0.779205   
2  010_190℃_020_0.5mm.csv   0.5          20         0.779205   
3  010_190℃_020_0.5mm.csv   0.5          20         0.779205   
4  010_190℃_020_0.5mm.csv   0.5          20         0.779205   

   せん断速度γ34(1/s)_幾何標準偏差下  樹脂粘度η34(Pa・s)_幾何平均  樹脂粘度η34(Pa・s)_幾何標準偏差上  \
0            3003.003003           64.188324              64.188324   
1            3003.003003           64.188324              64.188324   
2            3003.003003           64.188324              64.188324   
3            3003.003003           64.188324              64.188324   
4            3003.003003           64.188324              64.188324   

   樹脂粘度η34(Pa・s)_幾何標準偏差下  
0              64.188324  
1              64.188324  
2              64.188324  
3              64.188324  
4        

In [5]:
# =====================================================================================
# セル4: 計算済みデータのCSV保存
# =====================================================================================

if 'dataframes' in locals() and dataframes:
    print("\n--- ステップ4: 計算結果のCSVファイル保存 ---")
    output_folder = os.path.dirname(file_paths[0])
    
    for original_filename, df_to_save in dataframes.items():
        base_name = os.path.splitext(original_filename)[0]
        new_filename = f"{base_name}_processed.csv"
        output_path = os.path.join(output_folder, new_filename)
        try:
            df_to_save.to_csv(output_path, index=False, encoding='utf-8-sig')
            print(f"✅ 保存成功: {output_path}")
        except Exception as e:
            print(f"❌ 保存失敗: {output_path} - {e}")


--- ステップ4: 計算結果のCSVファイル保存 ---
✅ 保存成功: C:/Users/0uh2j/OneDrive/Desktop/実験データ2026/本実験データ/算出グラフ_ローパス条件比較/カットオフ周波数80閾値0.4\NR_summary_rise_start_results_processed.csv


In [6]:
# =====================================================================================
# セル5: 【設定】グラフのレイアウトと軸範囲
# =====================================================================================

CV_THICKNESS_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*'] 
TEMP_MARKERS = {'190℃': 'o', '210℃': 's', '230℃': '^'}
HORIZONTAL_GRAPH_GAP = 10

FIGSIZE_NORMAL = (11, 6)        
FIGSIZE_LOGLOG = (12, 8)        
FIGSIZE_SUMMARY = (12, 6)       
FIGSIZE_LOGLOG_SUMMARY = (12, 8)

LINE_MARKER_SIZE = 10      
SCATTER_MARKER_SIZE = 90  

MANUAL_AXIS_LIMITS = {
    '射出速度と時間差の関係': {
    # 2026-10-03: 縦軸を0〜0.6 → 0〜1.2 sに変更（1.5mm・2.0mmの低速条件が範囲外になっていたため）
             0: {'xlim': (0, 330), 'ylim': (0, 1.2), 'xinterval': 5, 'yinterval': 0.1},
             1: {'xlim': (0, 330), 'ylim': (0, 1.2), 'xinterval': 5, 'yinterval': 0.1},
             2: {'xlim': (0, 330), 'ylim': (0, 1.2), 'xinterval': 5, 'yinterval': 0.1},
             3: {'xlim': (0, 330), 'ylim': (0, 1.2), 'xinterval': 5, 'yinterval': 0.1},
     'all_data': {'xlim': (0, 330), 'ylim': (0, 1.2), 'xinterval': 5, 'yinterval': 0.1},
    },
    '射出速度と流速の関係': {
        'default': {'xlim': (0, 350), 'ylim': (0, 5000), 'xinterval': 5, 'yinterval': 500},
        'all_data': {'xlim': (0, 350), 'ylim': (0, 5000), 'xinterval': 5, 'yinterval': 500},
    },
    '射出速度と流量の関係': {
        'default': {'xlim': (0, 350), 'ylim': (0, 300000), 'xinterval': 5, 'yinterval': 50000},
        'all_data': {'xlim': (0, 350), 'ylim': (0, 300000), 'xinterval': 5, 'yinterval': 50000},
    },
    '射出速度と圧力差の関係': {
        'default': {'xlim': (0, 350), 'ylim': (0, 3 * 10**7), 'xinterval': 5, 'yinterval': 0.5 * 10**7},
        'all_data': {'xlim': (0, 350), 'ylim': (0, 3*10**7), 'xinterval': 5, 'yinterval': 0.5*10**7},
    },
    'せん断速度と樹脂粘度の関係': {
        # 2026-10-03: 01〜05で統一。縦軸下限を0.1に変更（0.5mmの高速条件が1 Pa・sを下回るため）
        'default': {'xlim': (10**0, 10**5), 'ylim': (10**-1, 3 * 10**4), 'xinterval': None, 'yinterval': None},
        'all_data': {'xlim': (10**0, 10**5), 'ylim': (10**-1, 3 * 10**4), 'xinterval': None, 'yinterval': None},
    }
}

print("グラフの全体スタイル設定を読み込みました。")

グラフの全体スタイル設定を読み込みました。


In [7]:
# =====================================================================================
# セル6・7: 【設定】総合比較グラフのスタイルおよびグラデーション設定
# =====================================================================================

CAVITY_THICKNESS_GRADIENT_SETTINGS = {
    0.5: {'start_color': 'red', 'end_color': 'pink'}, 
    1.0: {'start_color': 'blue', 'end_color': 'lightblue'}, 
    1.5: {'start_color': 'gold', 'end_color': 'yellow'}, 
    2.0: {'start_color': 'green', 'end_color': 'lightgreen'}, 
}

print("総合比較グラフのスタイル設定を読み込みました。")

総合比較グラフのスタイル設定を読み込みました。


In [8]:
# =====================================================================================
# セル8: グラフ作成ツールの定義（個別/総合）
# =====================================================================================

print("\n--- グラフ作成の準備 ---")
plt.rcParams['xtick.direction'] = 'in'; plt.rcParams['ytick.direction'] = 'in'

def set_japanese_font():
    standard_fonts = ['Yu Gothic', 'Meiryo', 'MS Gothic', 'Hiragino Sans', 'IPAexGothic']
    for font in standard_fonts:
        try:
            plt.rcParams['font.family'] = font; plt.rcParams['axes.unicode_minus'] = False; return
        except: continue
set_japanese_font()

class GraphCounter:
    def __init__(self): self.count = 1
    def get_next(self): num = self.count; self.count += 1; return num

def create_safe_filename(prefix, suffix, extension="png"):
    return f"{re.sub(r'[\\/:*?\"<>|]', '_', prefix)}_{re.sub(r'[\\/:*?\"<>|]', '_', suffix)}.{extension}"

def generate_color_gradient(start_color, end_color, num_steps):
    from matplotlib.colors import to_rgba
    if num_steps <= 1: return [to_rgba(start_color)]
    start_rgba = np.array(to_rgba(start_color)); end_rgba = np.array(to_rgba(end_color))
    return [tuple(start_rgba + (end_rgba - start_rgba) * i / (num_steps - 1)) for i in range(num_steps)]

def set_manual_log_ticks(ax, xlims, ylims):
    if xlims and xlims[0] > 0 and xlims[1] > 0:
        ticks_x = [10**i for i in range(int(np.floor(np.log10(xlims[0]))), int(np.ceil(np.log10(xlims[1]))) + 1)]
        ax.set_xticks(ticks_x)
    if ylims and ylims[0] > 0 and ylims[1] > 0:
        ticks_y = [10**i for i in range(int(np.floor(np.log10(ylims[0]))), int(np.ceil(np.log10(ylims[1]))) + 1)]
        ax.set_yticks(ticks_y)

# --- 通常グラフ作成関数 ---
def create_linear_graphs_for_thickness(df_thick, workbook, worksheet, x_col, y_col, start_row, color_map, counter, thickness_label, save_folder_path, xlims, ylims, xinterval, yinterval):
    graph_title = f'{y_col}の比較'
    worksheet.set_row(start_row, 294)
    unique_temps = sorted(df_thick['温度'].unique())
    
    # 総合グラフ (エラーバー無)
    fig_sum, ax_sum = plt.subplots(figsize=FIGSIZE_SUMMARY)
    ax_sum.set_title(f"[総合] {graph_title}\n(CV厚: {thickness_label}mm)")
    ax_sum.set_xlabel(x_col); ax_sum.set_ylabel(y_col); ax_sum.grid(True)
    if xlims: ax_sum.set_xlim(xlims); 
    if ylims: ax_sum.set_ylim(ylims)
    if xinterval: 
        ax_sum.set_xticks(np.arange(xlims[0], xlims[1] + xinterval, xinterval))
        ax_sum.tick_params(axis='x', rotation=45)
        
    current_col = 1
    for temp in unique_temps:
        df_temp = df_thick[df_thick['温度'] == temp]
        curve_data = df_temp.groupby('射出速度[mm/s]', as_index=False).first().sort_values('射出速度[mm/s]')
        
        x = curve_data['射出速度[mm/s]']
        y = curve_data[f'{y_col}_平均']
        yerr = curve_data[f'{y_col}_標準偏差上'] # 絶対値で取得
        color = color_map.get(f"{int(temp)}℃", 'black')
        
        # 総合グラフ描画
        ax_sum.plot(x, y, marker=TEMP_MARKERS.get(f"{int(temp)}℃", 'o'), linestyle='-', color=color, label=f"{int(temp)}℃")
        
        # 個別グラフ (エラーバー有)
        fig_ind, ax_ind = plt.subplots(figsize=FIGSIZE_NORMAL)
        serial_num = counter.get_next()
        ax_ind.errorbar(x, y, yerr=yerr, marker=TEMP_MARKERS.get(f"{int(temp)}℃", 'o'), linestyle='-', color=color, capsize=4, label=f"{int(temp)}℃")
        ax_ind.set_title(f"[{serial_num}] 個別: {graph_title}\n(CV厚: {thickness_label}mm / {int(temp)}℃)")
        ax_ind.set_xlabel(x_col); ax_ind.set_ylabel(y_col); ax_ind.grid(True)
        ax_ind.legend(loc='best', frameon=False, fontsize='large')
        if xlims: ax_ind.set_xlim(xlims); 
        if ylims: ax_ind.set_ylim(ylims)
        if xinterval:
            ax_ind.set_xticks(np.arange(xlims[0], xlims[1] + xinterval, xinterval))
            ax_ind.tick_params(axis='x', rotation=45)
        
        fig_ind.tight_layout()
        img_filename = create_safe_filename(f"[{serial_num}]_CV{thickness_label}_ind", f"{y_col}_{int(temp)}C")
        fig_ind.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
        img_data = io.BytesIO(); fig_ind.savefig(img_data, format='png'); plt.close(fig_ind)
        worksheet.write(start_row - 1, current_col, f"[{serial_num}] {int(temp)}℃ 個別(エラーバー有)")
        worksheet.insert_image(start_row, current_col, '', {'image_data': img_data})
        current_col += 12 + HORIZONTAL_GRAPH_GAP
        
    ax_sum.legend(loc='best', frameon=False, fontsize='large')
    fig_sum.tight_layout()
    serial_num_sum = counter.get_next()
    img_filename_sum = create_safe_filename(f"[{serial_num_sum}]_CV{thickness_label}_sum", y_col)
    fig_sum.savefig(os.path.join(save_folder_path, img_filename_sum), dpi=150)
    img_data_sum = io.BytesIO(); fig_sum.savefig(img_data_sum, format='png'); plt.close(fig_sum)
    worksheet.write(start_row - 1, current_col, f"[{serial_num_sum}] 総合比較(エラーバー無)")
    worksheet.insert_image(start_row, current_col, '', {'image_data': img_data_sum})

# --- 対数グラフ作成関数 ---
def create_loglog_graphs_for_thickness(df_thick, workbook, worksheet, x_col, y_col, start_row, color_map, counter, thickness_label, save_folder_path, xlims, ylims):
    graph_title = f'{y_col}の比較'
    worksheet.set_row(start_row, 375)
    unique_temps = sorted(df_thick['温度'].unique())
    
    fig_sum, ax_sum = plt.subplots(figsize=FIGSIZE_LOGLOG_SUMMARY)
    ax_sum.set_title(f"[総合] {graph_title}\n(CV厚: {thickness_label}mm)")
    ax_sum.set_xlabel('せん断速度 (1/s)'); ax_sum.set_ylabel('樹脂粘度 (Pa・s)'); ax_sum.grid(False)
    ax_sum.set_xscale('log'); ax_sum.set_yscale('log')
    if xlims: ax_sum.set_xlim(xlims)
    if ylims: ax_sum.set_ylim(ylims)
    
    current_col = 1
    for temp in unique_temps:
        df_temp = df_thick[df_thick['温度'] == temp]
        curve_data = df_temp.groupby('射出速度[mm/s]', as_index=False).first().sort_values('射出速度[mm/s]')
        
        x = curve_data[f'{x_col}_幾何平均']; y = curve_data[f'{y_col}_幾何平均']
        # 幾何標準偏差からエラーバーの距離を算出
        yerr = [y - curve_data[f'{y_col}_幾何標準偏差下'], curve_data[f'{y_col}_幾何標準偏差上'] - y]
        xerr = [x - curve_data[f'{x_col}_幾何標準偏差下'], curve_data[f'{x_col}_幾何標準偏差上'] - x]
        
        color = color_map.get(f"{int(temp)}℃", 'black')
        marker = TEMP_MARKERS.get(f"{int(temp)}℃", 'o')
        
        ax_sum.plot(x, y, marker=marker, linestyle='-', color=color, label=f"{int(temp)}℃")
        
        fig_ind, ax_ind = plt.subplots(figsize=FIGSIZE_LOGLOG)
        serial_num = counter.get_next()
        ax_ind.errorbar(x, y, xerr=xerr, yerr=yerr, marker=marker, linestyle='-', color=color, capsize=4, label=f"{int(temp)}℃")
        ax_ind.set_title(f"[{serial_num}] 個別: {graph_title}\n(CV厚: {thickness_label}mm / {int(temp)}℃)")
        ax_ind.set_xlabel('せん断速度 (1/s)'); ax_ind.set_ylabel('樹脂粘度 (Pa・s)'); ax_ind.grid(False)
        ax_ind.set_xscale('log'); ax_ind.set_yscale('log')
        ax_ind.legend(loc='best', frameon=False, fontsize='large')
        if xlims: ax_ind.set_xlim(xlims)
        if ylims: ax_ind.set_ylim(ylims)
        
        fig_ind.tight_layout()
        img_filename = create_safe_filename(f"[{serial_num}]_CV{thickness_label}_log_ind", f"{y_col}_{int(temp)}C")
        fig_ind.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
        img_data = io.BytesIO(); fig_ind.savefig(img_data, format='png'); plt.close(fig_ind)
        worksheet.write(start_row - 1, current_col, f"[{serial_num}] {int(temp)}℃ 個別(幾何エラーバー有)")
        worksheet.insert_image(start_row, current_col, '', {'image_data': img_data})
        current_col += 15 + HORIZONTAL_GRAPH_GAP
        
    ax_sum.legend(loc='best', frameon=False, fontsize='large')
    fig_sum.tight_layout()
    serial_num_sum = counter.get_next()
    img_filename_sum = create_safe_filename(f"[{serial_num_sum}]_CV{thickness_label}_log_sum", y_col)
    fig_sum.savefig(os.path.join(save_folder_path, img_filename_sum), dpi=150)
    img_data_sum = io.BytesIO(); fig_sum.savefig(img_data_sum, format='png'); plt.close(fig_sum)
    worksheet.write(start_row - 1, current_col, f"[{serial_num_sum}] 総合比較(エラーバー無)")
    worksheet.insert_image(start_row, current_col, '', {'image_data': img_data_sum})

print("エラーバー対応版のグラフ作成関数を定義しました。")


--- グラフ作成の準備 ---
エラーバー対応版のグラフ作成関数を定義しました。


In [9]:
# =====================================================================================
# セル9: 全データ総集編グラフ作成（エラーバーなし）
# =====================================================================================

def create_final_all_data_graph(df_full, workbook, worksheet, x_col, y_col, start_row, color_map, counter, save_folder_path, start_col, xlims, ylims, xinterval=None, yinterval=None, is_log=False):
    serial_num = counter.get_next()
    graph_title = f'全データ総合比較: {y_col}'
    fig, ax = plt.subplots(figsize=FIGSIZE_SUMMARY if not is_log else FIGSIZE_LOGLOG_SUMMARY)
    
    unique_thicknesses = sorted(df_full['キャビ厚'].unique())
    unique_temps = sorted(df_full['温度'].unique())
    
    for i, thickness_val in enumerate(unique_thicknesses):
        df_thick = df_full[df_full['キャビ厚'] == thickness_val]
        marker = CV_THICKNESS_MARKERS[i % len(CV_THICKNESS_MARKERS)]
        
        for temp in unique_temps:
            df_temp = df_thick[df_thick['温度'] == temp]
            if df_temp.empty: continue
            curve_data = df_temp.groupby('射出速度[mm/s]', as_index=False).first().sort_values('射出速度[mm/s]')
            
            if is_log:
                ax.plot(curve_data[f'{x_col}_幾何平均'], curve_data[f'{y_col}_幾何平均'], marker=marker, linestyle='-', color=color_map.get(f"{int(temp)}℃", 'black'))
            else:
                ax.plot(curve_data['射出速度[mm/s]'], curve_data[f'{y_col}_平均'], marker=marker, linestyle='-', color=color_map.get(f"{int(temp)}℃", 'black'))
                
    ax.set_title(f"[{serial_num}] {graph_title}")
    if is_log:
        ax.set_xlabel('せん断速度 (1/s)'); ax.set_ylabel('樹脂粘度 (Pa・s)'); ax.grid(False)
        ax.set_xscale('log'); ax.set_yscale('log')
        set_manual_log_ticks(ax, xlims, ylims)
    else:
        ax.set_xlabel('射出速度[mm/s]'); ax.set_ylabel(y_col); ax.grid(True)
        if xinterval: 
            ax.set_xticks(np.arange(xlims[0], xlims[1] + xinterval, xinterval))
            ax.tick_params(axis='x', rotation=45)
        if yinterval: ax.set_yticks(np.arange(ylims[0], ylims[1] + yinterval, yinterval))
    if xlims: ax.set_xlim(xlims)
    if ylims: ax.set_ylim(ylims)
    
    # 凡例の作成
    from matplotlib.lines import Line2D
    dummy = Line2D([0], [0], marker='none', color='none', linestyle='none')
    temp_h = [Line2D([0], [0], marker='o', color=color_map.get(f"{int(t)}℃", 'k'), linestyle='-') for t in unique_temps]
    thick_h = [Line2D([0], [0], marker=CV_THICKNESS_MARKERS[i%len(CV_THICKNESS_MARKERS)], color='gray', linestyle='None') for i, t in enumerate(unique_thicknesses)]
    legend = ax.legend([dummy]+temp_h+[dummy]+thick_h, ['Temperature']+[f"{int(t)}℃" for t in unique_temps]+['Cavity Thickness']+[f"{t}mm" for t in unique_thicknesses], loc='best', frameon=False)
    for text in legend.get_texts():
        if text.get_text() in ['Temperature', 'Cavity Thickness']: text.set_weight('bold')
            
    fig.tight_layout()
    img_filename = create_safe_filename(f"[{serial_num}]_final_all", y_col)
    fig.savefig(os.path.join(save_folder_path, img_filename), dpi=150)
    img_data = io.BytesIO(); fig.savefig(img_data, format='png'); plt.close(fig)
    
    worksheet.write(start_row - 1, start_col, f"[{serial_num}] 全データ総合比較(エラーバー無)")
    worksheet.insert_image(start_row, start_col, '', {'image_data': img_data})
    return 12 if not is_log else 15

In [10]:
# =====================================================================================
# セル10: Excelレポートとグラフ画像の自動生成
# =====================================================================================

if 'file_paths' in locals() and file_paths:
    print("\n--- Excelレポートの生成を開始します ---")
    output_folder = os.path.dirname(file_paths[0])
    
    for target_key, df in dataframes.items():
        graph_counter = GraphCounter()
        base_name = os.path.splitext(target_key)[0]
        main_graph_folder = os.path.join(output_folder, f"graph_images_{base_name}")
        os.makedirs(main_graph_folder, exist_ok=True)
        excel_output_path = os.path.join(output_folder, f"graph_output_{base_name}.xlsx")

        with pd.ExcelWriter(excel_output_path, engine='xlsxwriter') as writer:
            workbook = writer.book
            print(f"対象ファイル: {target_key}")

            unique_temps = sorted(df['温度'].dropna().unique())
            colors = ['blue', 'green', 'red', 'cyan', 'magenta', 'yellow', 'black', 'purple']
            color_map = {f"{int(t)}℃": colors[i % len(colors)] for i, t in enumerate(unique_temps)}
            
            sheet_definitions = {
                '射出速度と時間差の関係': ['時間差ΔT34(s)'],
                '射出速度と流速の関係': ['流速V34(mm/s)'],
                '射出速度と流量の関係': ['流量Q34(mm³/s)'],
                '射出速度と圧力差の関係': ['圧力差ΔP34(Pa)']
            }
            log_sheet_def = {
                'せん断速度と樹脂粘度の関係': [('せん断速度γ34(1/s)', '樹脂粘度η34(Pa・s)')]
            }
            
            all_sheet_defs = {**sheet_definitions, **log_sheet_def}
            unique_thicknesses = sorted(df['キャビ厚'].dropna().unique())
            
            for sheet_name, cols in all_sheet_defs.items():
                worksheet = workbook.add_worksheet(sheet_name)
                current_row = 1
                current_save_path = os.path.join(main_graph_folder, re.sub(r'[\\/:*?"<>|]', '_', sheet_name))
                os.makedirs(current_save_path, exist_ok=True)
                
                # --- セクション1: キャビティ厚みごとのグラフ（個別エラーバー＋厚み別総合） ---
                for thickness_val, df_thick in df.groupby('キャビ厚'):
                    worksheet.merge_range(current_row, 0, current_row, 40, f"--- キャビティ厚み: {thickness_val}mm ---", workbook.add_format({'bold': True, 'align': 'center', 'bg_color': '#DDEBF7'}))
                    current_row += 2
                    
                    chunk_index = unique_thicknesses.index(thickness_val)
                    limits = {}
                    sheet_settings = MANUAL_AXIS_LIMITS.get(sheet_name, {})
                    chunk_specific_settings = sheet_settings.get(chunk_index, sheet_settings.get('default', {}))
                    xlim = chunk_specific_settings.get('xlim')
                    ylim = chunk_specific_settings.get('ylim')
                    
                    if sheet_name in sheet_definitions:
                        for y_col in cols:
                            create_linear_graphs_for_thickness(df_thick, workbook, worksheet, '射出速度[mm/s]', y_col, current_row, color_map, graph_counter, thickness_val, current_save_path, xlim, ylim, chunk_specific_settings.get('xinterval'), chunk_specific_settings.get('yinterval'))
                            current_row += 20
                    elif sheet_name in log_sheet_def:
                        for x_col, y_col in cols:
                            create_loglog_graphs_for_thickness(df_thick, workbook, worksheet, x_col, y_col, current_row, color_map, graph_counter, thickness_val, current_save_path, xlim, ylim)
                            current_row += 20
                    current_row += 3
                
                # --- セクション2: 全データ総集編グラフ（全厚み・全温度を重ねる／エラーバー無） ---
                worksheet.merge_range(current_row, 0, current_row, 40, "--- 全データ総集編グラフ ---", workbook.add_format({'bold': True, 'align': 'center', 'bg_color': '#BFBFBF'}))
                current_row += 2
                final_graph_col = 1
                
                final_limits = MANUAL_AXIS_LIMITS.get(sheet_name, {}).get('all_data', {})
                if sheet_name in sheet_definitions:
                    for y_col in cols:
                        span = create_final_all_data_graph(df, workbook, worksheet, '射出速度[mm/s]', y_col, current_row, color_map, graph_counter, current_save_path, final_graph_col, final_limits.get('xlim'), final_limits.get('ylim'), final_limits.get('xinterval'), final_limits.get('yinterval'), is_log=False)
                        final_graph_col += span + HORIZONTAL_GRAPH_GAP
                elif sheet_name in log_sheet_def:
                    for x_col, y_col in cols:
                        span = create_final_all_data_graph(df, workbook, worksheet, x_col, y_col, current_row, color_map, graph_counter, current_save_path, final_graph_col, final_limits.get('xlim'), final_limits.get('ylim'), is_log=True)
                        final_graph_col += span + HORIZONTAL_GRAPH_GAP
                        
            print(f"\n--- ファイル '{target_key}' の処理が完了しました ---")


--- Excelレポートの生成を開始します ---


対象ファイル: NR_summary_rise_start_results.csv


C:\Users\0uh2j\AppData\Local\Temp\ipykernel_26096\2294262424.py:106: UserWarning: Attempt to set non-positive xlim on a log-scaled axis will be ignored.
  if xlims: ax_sum.set_xlim(xlims)
C:\Users\0uh2j\AppData\Local\Temp\ipykernel_26096\2294262424.py:131: UserWarning: Attempt to set non-positive xlim on a log-scaled axis will be ignored.
  if xlims: ax_ind.set_xlim(xlims)
C:\Users\0uh2j\AppData\Local\Temp\ipykernel_26096\2886385186.py:38: UserWarning: Attempt to set non-positive xlim on a log-scaled axis will be ignored.
  if xlims: ax.set_xlim(xlims)



--- ファイル 'NR_summary_rise_start_results.csv' の処理が完了しました ---
